### Step 1:Initial Setup: Installs the required packages

In [2]:
!pip install -q torch transformers datasets accelerate scikit-learn pandas matplotlib seaborn

### Step 2: Imports the libraries

In [3]:
import os # Facilitates working with folders and filepaths
import re # Search text using patterns, used for skill matching
import numpy as np 
import pandas as pd
import torch # Main PyTorch library for deep learning
import torch.nn as nn # Used for neural network tools like loss functions

from datasets import load_dataset # Loads datasets from Hugging Face
from transformers import ( # Imports tools from Hugging Face Transformers
    AutoTokenizer, # Loads the appropriate tokenizer for the model
    AutoModelForSequenceClassification, # Loads a transformer model used for classification
    TrainingArguments, # Used to store training settings like epochs and batch size
    Trainer, # Used to handle training and evaluation loop
    DataCollatorWithPadding # Used to pad batches so inputs have matching lengths
)

from sklearn.metrics import ( # Imports evaluation metrics from scikit-learn
    accuracy_score, # Used to calculate overall accuracy of the prediction
    precision_recall_fscore_support, # Used to calculate precision, recall, and F1-score
    classification_report, # Used to create a detailed performance report by class
    confusion_matrix # Used to identify where model is making incorrect and correct predictions
)

from sklearn.utils.class_weight import compute_class_weight # Used to create class weights to handle label imbalances

### Step 3: Checks if GPU is available

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu" # Use GPU if available, otherwise use CPU

print("Notebook will use this device:", device) # Prints whether GPU or CPU is being used
print("CUDA available for use:", torch.cuda.is_available()) # Checks for CUDA/GPU support

if torch.cuda.is_available(): # Runs if CUDA-compatible GPU is found
    print("GPU being used is:", torch.cuda.get_device_name(0)) # Prints the name of the GPU identified

Notebook will use this device: cuda
CUDA available for use: True
GPU being used is: NVIDIA GeForce RTX 4060 Laptop GPU


### Step 4:  Loads the Hugging Face dataset

In [5]:
dataset = load_dataset("cnamuangtoun/resume-job-description-fit") # Loads the resume-job description dataset from Hugging Face

print(dataset) # Prints the dataset structure, including number of rows and train/test splits

print(dataset["train"].column_names) # Prints the column names in the training dataset

DatasetDict({
    train: Dataset({
        features: ['resume_text', 'job_description_text', 'label'],
        num_rows: 6241
    })
    test: Dataset({
        features: ['resume_text', 'job_description_text', 'label'],
        num_rows: 1759
    })
})
['resume_text', 'job_description_text', 'label']


### Step 5: Previews the data

In [6]:
train_df = pd.DataFrame(dataset["train"]) # Converts the training split into a pandas DataFrame table

print(train_df.head()) # Print the first five rows

print(train_df["label"].value_counts()) # Counts the number of examples in each label/class

                                         resume_text  \
0  SummaryHighly motivated Sales Associate with e...   
1  Professional SummaryCurrently working with Cat...   
2  SummaryI started my construction career in Jun...   
3  SummaryCertified Electrical Foremanwith thirte...   
4  SummaryWith extensive experience in business/r...   

                                job_description_text   label  
0  Net2Source Inc. is an award-winning total work...  No Fit  
1  At Salas OBrien we tell our clients that were ...  No Fit  
2  Schweitzer Engineering Laboratories (SEL) Infr...  No Fit  
3  Mizick Miller & Company, Inc. is looking for a...  No Fit  
4  Life at Capgemini\nCapgemini supports all aspe...  No Fit  
label
No Fit           3143
Potential Fit    1556
Good Fit         1542
Name: count, dtype: int64


### Step 6: Creates train, validation, and test datasets. The dataset already has train and test splits. However, we still need to create a small validation set from the training data.


In [7]:
train_valid_split = dataset["train"].train_test_split(test_size = 0.1, seed = 52) # 90% Training and 10% validation, repeatable splits

train_dataset = train_valid_split["train"] # Stores the training dataset
valid_dataset = train_valid_split["test"] # Stores the validation dataset
test_dataset = dataset["test"] # Uses the datasets original test split for the final evaluation

print("The Training dataset size is:", len(train_dataset)) # Prints the number of training examples
print("The Validation dataset size is:", len(valid_dataset)) # Prints the number of validation examples
print("The Test dataset size is:", len(test_dataset)) # Print the number of test examples

The Training dataset size is: 5616
The Validation dataset size is: 625
The Test dataset size is: 1759


### Step 7: Prepares label mapping

In [8]:
# Gets unique labels from the dataset
unique_labels = sorted(list(set(train_dataset["label"]))) # Sorts unique labels from dataset alphabetically

print("The original labels are:", unique_labels) # Prints the unique labels

preferred_order = ["No Fit", "Potential Fit", "Good Fit"] # Setting the label order to be used

if all(label in unique_labels for label in preferred_order): # Checks if preferred labels match those in the dataset
    label_names = preferred_order # Assigned the preferred labels
else:
    label_names = unique_labels # Use dataset's labels if preferred labels are missing

label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for label, i in label2id.items()}

print("The label to id mapping is:", label2id) # Prints the label to number mapping
print("The id to label mapping is:", id2label) # Checks model prediction labels match input label mappings

The original labels are: ['Good Fit', 'No Fit', 'Potential Fit']
The label to id mapping is: {'No Fit': 0, 'Potential Fit': 1, 'Good Fit': 2}
The id to label mapping is: {0: 'No Fit', 1: 'Potential Fit', 2: 'Good Fit'}


### Step 8: Converts text labels to numbers

In [9]:
def encode_labels(example): # Define the function used to convert text labels to numeric labels
    example["labels"] = label2id[example["label"]] # Looks up the number for the current text label
    return example # Returns the updated example with new numeric label column

train_dataset = train_dataset.map(encode_labels) # Applies the label conversion to the training dataset
valid_dataset = valid_dataset.map(encode_labels) # Applies the label conversion to the validation dataset
test_dataset = test_dataset.map(encode_labels)   # Applies the label conversion to the test dataset

train_labels = train_dataset["labels"] # Stores the numeric training labels, used for calculating the class weights

class_weights = compute_class_weight( # Calculates the weights for all classes, to handle label distribution imbalances
    class_weight="balanced", # Assigns higher weight to classes with smaller distributions
    classes=np.array([0, 1, 2]), # Lists the numeric class labels
    y=np.array(train_labels) # Provides actual training labels for weight calculation
)

class_weights = torch.tensor(class_weights, dtype=torch.float) # Converts class weights into a PyTorch tensor

print("The class weights are:", class_weights) # Prints the class weights for inspection

Map:   0%|          | 0/5616 [00:00<?, ? examples/s]

Map:   0%|          | 0/625 [00:00<?, ? examples/s]

The class weights are: tensor([0.6622, 1.3295, 1.3555])


### Step 9: Chooses transformer model

In [10]:
model_checkpoint = "distilbert-base-uncased" # Using the pre-trained DistilBERT model

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint) # Loads the correct tokenizer for DistilBERT

### Step 10: Tokenizes resume and job description

In [11]:
def encode_and_tokenize_function(example): # Defines the function to tokenize one resume-job pair
    tokenized = tokenizer( # Uses the DistilBERT tokenizer to convert text into inputs for the model
        example["resume_text"], # Resume text input
        example["job_description_text"], # Job description text input
        truncation=True, # Limits the text to maximum text length by cutting excess words
        max_length=512 # Defines maximum length for each resume-job pair
    )
    
    tokenized["labels"] = label2id[example["label"]] # Provides the numerical label for training
    
    return tokenized # Returns the tokenized input and labels


train_tokenized = train_dataset.map( # Tokenization applied on the training dataset
    encode_and_tokenize_function, # Uses previously defined tokenization function
    remove_columns=train_dataset.column_names # Removes the text columns
)

valid_tokenized = valid_dataset.map( # Tokenization applied on the validation dataset
    encode_and_tokenize_function, # Uses previously defined tokenization function
    remove_columns=valid_dataset.column_names # Removes the text columns
)

test_tokenized = test_dataset.map( # Tokenization applied on the test dataset
    encode_and_tokenize_function, # Uses previously defined tokenization function
    remove_columns=test_dataset.column_names # Removes the text columns
)

print(train_tokenized.column_names) # Prints the column names after tokenization
print(train_tokenized[0]) # Prints the first tokenized training example

Map:   0%|          | 0/5616 [00:00<?, ? examples/s]

Map:   0%|          | 0/625 [00:00<?, ? examples/s]

['labels', 'input_ids', 'token_type_ids', 'attention_mask']
{'labels': 2, 'input_ids': [101, 3325, 12260, 22601, 2389, 3992, 1010, 2340, 1013, 2418, 3406, 10841, 14343, 3372, 6914, 21673, 3751, 1516, 6243, 15951, 1010, 6187, 1010, 6509, 2007, 1996, 2458, 1997, 7661, 5992, 2491, 2291, 5997, 3688, 2109, 1999, 3919, 3684, 2121, 3001, 1999, 10388, 2000, 17359, 2753, 2620, 2050, 1998, 24940, 5174, 2581, 2683, 4781, 1012, 2640, 1997, 5992, 2373, 13782, 1998, 3001, 2109, 1999, 2491, 9320, 1012, 4339, 4031, 12827, 5491, 2164, 4488, 2373, 4009, 1013, 9829, 16268, 1010, 1998, 4009, 5992, 8040, 28433, 14606, 2478, 5024, 9316, 5992, 4007, 1012, 2490, 5814, 1998, 3737, 2491, 1997, 5992, 2491, 9320, 2164, 3320, 1998, 10569, 1997, 3115, 1998, 7661, 3688, 1012, 2023, 2950, 4488, 8360, 1998, 9829, 5604, 2006, 2536, 4127, 1997, 2491, 9320, 2011, 2770, 2613, 1011, 2088, 16820, 2007, 7170, 5085, 1010, 2951, 3405, 4007, 1010, 1998, 3223, 3941, 1012, 8139, 3992, 1010, 5718, 1013, 2286, 3406, 24096, 1013, 23

### Step 11: Creates the transformer classification model. Starts with the pre-trained DistilBERT model, then adds a classification layer in order to predict the resume-job description fit labels.

In [12]:
num_labels = len(label_names) # Counts the number of labels model would need to predict

model = AutoModelForSequenceClassification.from_pretrained( # Loads the pre-trained transformer model for classification
    model_checkpoint, # Uses the model name stored earlier
    num_labels=num_labels, # Gives the model number of output classes
    id2label=id2label, # Model can now convert numeric predictions into label names 
    label2id=label2id # Model can now also convert label names into numeric predictions
)

model.to(device) # Model uses GPU if available, otherwise uses CPU

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

### Step 12: Defines evaluation metrics, model outputs raw scores or logits, which are then converted into predicted classes using np.argmax.

In [13]:
def compute_metrics(eval_pred): # Defines the function to calculate the model's evaluation scores
    logits, labels = eval_pred # Separating the model's raw outputs and the true labels
    predictions = np.argmax(logits, axis=1) # Selects the class with the highest score as the prediction

    accuracy = accuracy_score(labels, predictions) # Percentage of correct predictions calculated

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support( # Calculates the weighted precision, recall, and F1 
        labels, # True Labels
        predictions, # Model's Predictions
        average="weighted", # Classes weighted by number of examples
        zero_division=0 # Prevents errors if a class has zero predictions
    )

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support( # This calculates the macro-precision, recall, and F1
        labels, # True Labels
        predictions, # Model's Predictions
        average="macro", # All classes treated equally regardless of class size
        zero_division=0 # Prevents errors if a class has zero predictions
    )

    return { # Returns a dictionary with all the evaluation scores
        "accuracy": accuracy, # Accuracy which is the overall correctness score
        "precision_weighted": precision_weighted, # Precision score across classes, weighted by class size
        "recall_weighted": recall_weighted, # Recall score across classes, weighted by class size
        "f1_weighted": f1_weighted, # F1-score across classes, weighted by class size
        "precision_macro": precision_macro, # Macro precision score
        "recall_macro": recall_macro, # Macro recall score
        "f1_macro": f1_macro # Macro F1-score
    }

### Step 13: Training rules set for the transformer model.

In [14]:
training_args = TrainingArguments( 
    output_dir="./outputs/transformer_results", # Folder to save the model checkpoints and training outputs
    eval_strategy="epoch", # Evaluation is run at the end of every epoch
    save_strategy="epoch", # Model checkpoint is saved at the end of every epoch
    learning_rate=1e-5, # Controls how quickly model updates its weights during training
    per_device_train_batch_size=8, # Number of training examples that are processed at once on a device
    per_device_eval_batch_size=8, # Number of evaluation examples that are processed at once on a device
    num_train_epochs=8, # Number of full passes through the training dataset
    weight_decay=0.01, # Regularization is added to reduce overfitting
    warmup_ratio=0.1, # Learning rate gradually increased during the first 10% of training
    logging_steps=50, # Training logs printed every 50 steps
    load_best_model_at_end=True, # The best checkpoint reloaded after training
    metric_for_best_model="eval_loss", # Best model chosen based on validation loss
    greater_is_better=False, # Specifies lower validation loss is better
    report_to="none", # Ensures weights and biases are not sent to external-tracking tools
    fp16=torch.cuda.is_available() # Uses 16-bit numbers if CUDA GPU is available during training, for faster training
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


### Step 14: As No-Fit class is more common than the other two classes, the model may become more biased toward predicting it. This new custom trainer uses class weights, so smaller classes get more importance during training. 

In [15]:
# Weighted Trainer class

class WeightedLossTrainer(Trainer): # Creates a custom version of Hugging Face's trainer
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs): # **kwargs allows any extra keyword arguments to be accepted
        labels = inputs.get("labels") # Retrives correct labels for current batch
        outputs = model(**inputs) # Sends the input batch through the model, unpacking the dictionary automatically
        logits = outputs.get("logits") # Gets the model's raw prediction scores

        loss_function = nn.CrossEntropyLoss( # A classification loss function is created
            weight=class_weights.to(logits.device) # Class weights are applied and moved to the same device as the model
        )

        loss = loss_function( # Weighted classification loss is calculated
            logits.view(-1, model.config.num_labels), # Model outputs reshaped into expected format
            labels.view(-1) # Labels reshaped into expected format
        )

        return (loss, outputs) if return_outputs else loss # Returns loss and outputs or loss

### Step 15: Creates Training controller for the model. The data collator makes sure inputs are of the same size. WeightedLossTrainer brings together the model, training settings, training data, validation data, tokenizer, padding logic and evaluation metrics.

In [16]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer) # Creates the helper which pads each batch to the same length

trainer = WeightedLossTrainer( # Creates the custom trainer that uses weighted loss 
    model=model, # The DistilBERT classification model is being trained
    args=training_args, # The training settings are applied
    train_dataset=train_tokenized, # Teaches the model using tokenized training data
    eval_dataset=valid_tokenized, # Performance is checked on the validation data using tokenized validation data
    processing_class=tokenizer, # Tokenizer used passed on to trainer
    data_collator=data_collator, # Batches padded correctly before sending them to the model
    compute_metrics=compute_metrics # Calculates accuracy, precision, recall and F1 during evaluation
)

### Step 16: Trains the model

In [17]:
trainer.train() # Starts training the model using the trainer setup, training data and settings.

Epoch,Training Loss,Validation Loss,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted,Precision Macro,Recall Macro,F1 Macro
1,0.960852,0.943555,0.537600,0.541083,0.537600,0.511949,0.523941,0.512505,0.480687
2,0.839926,0.845016,0.595200,0.583058,0.595200,0.576867,0.563585,0.568911,0.549530
3,0.789696,0.805414,0.560000,0.641439,0.560000,0.537510,0.590266,0.635646,0.559851
4,0.668061,0.773513,0.620800,0.667217,0.620800,0.614148,0.621336,0.658603,0.612482
5,0.625297,0.749591,0.636800,0.678077,0.636800,0.633891,0.634378,0.674381,0.633345
6,0.597489,0.734677,0.664000,0.678761,0.664000,0.661283,0.645197,0.673155,0.648656
7,0.566062,0.746475,0.667200,0.694195,0.667200,0.665579,0.654430,0.691483,0.658876
8,0.518065,0.741834,0.681600,0.705030,0.681600,0.681632,0.666070,0.700324,0.672310


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5616, training_loss=0.7115549564021945, metrics={'train_runtime': 1137.0284, 'train_samples_per_second': 39.514, 'train_steps_per_second': 4.939, 'total_flos': 5951601423286272.0, 'train_loss': 0.7115549564021945, 'epoch': 8.0})

### Step 17: Evaluates on test set

In [18]:
test_results = trainer.evaluate(test_tokenized) # Trained model is evaluated on the test dataset

print(test_results) # Prints the final loss, accuracy, precision, recall and F1-score metrics

Training Loss,Validation Loss,Epoch,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted,Precision Macro,Recall Macro,F1 Macro
0.518065,1.418646,8,0.478681,0.440935,0.478681,0.444056,0.410254,0.395979,0.387768


{'eval_loss': 1.418646216392517, 'eval_accuracy': 0.47868106878908473, 'eval_precision_weighted': 0.44093543147745534, 'eval_recall_weighted': 0.47868106878908473, 'eval_f1_weighted': 0.44405648053300223, 'eval_precision_macro': 0.41025432229893194, 'eval_recall_macro': 0.39597868315136336, 'eval_f1_macro': 0.3877684525157646}


### Step 18: Gets predictions

In [19]:
predictions_output = trainer.predict(test_tokenized) # Uses the trained model for predictions on the test dataset

logits = predictions_output.predictions # Obtains raw prediction scores for each class
true_labels = predictions_output.label_ids # Gets the actual numeric labels from the test dataset
predicted_ids = np.argmax(logits, axis=1) # Class with highest score for each example is selected

predicted_labels = [id2label[i] for i in predicted_ids] # Predicted numeric labels changed into text
true_label_names = [id2label[i] for i in true_labels] # Converts actual numeric labels into text

print(classification_report(true_label_names, predicted_labels)) # Prints precision, recall, F1-score, and support for each class

               precision    recall  f1-score   support

     Good Fit       0.35      0.23      0.28       458
       No Fit       0.54      0.75      0.63       857
Potential Fit       0.33      0.20      0.25       444

     accuracy                           0.48      1759
    macro avg       0.41      0.40      0.39      1759
 weighted avg       0.44      0.48      0.44      1759



### Step 19: Confusion matrix

In [20]:
cm = confusion_matrix(true_label_names, predicted_labels, labels=label_names) # Confusion matrix to compare true labels vs predicted labels

cm_df = pd.DataFrame( # Confusion matrix converted into readable pandas table
    cm, # Confusion matrix values
    index=[f"Actual {label}" for label in label_names], # Row names with actual classes
    columns=[f"Predicted {label}" for label in label_names] # Column names with predicted classes
)

cm_df # Displays confusion matrix 

,Predicted No Fit,Predicted Potential Fit,Predicted Good Fit
Actual No Fit,646,97,114
Actual Potential Fit,274,90,80
Actual Good Fit,270,82,106


### Step 20: Converts predictions into fit scores and skill match scores

In [21]:
def softmax(x): # Defines a softmax function to convert raw scores into probabilities
    exp_x = np.exp(x - np.max(x)) # Exponentiation applied. Max value subtracted for maximum stability
    return exp_x / exp_x.sum() # Scores converted into probabilities which add up to 1

def calculate_fit_score(logit_row): # Defines a function to convert a prediction into a 0-100 fit score
    probabilities = softmax(logit_row) # Uses the models raw scores to get probabilities

    score_weights = { # Assigns the score to each fit category
        "No Fit": 0, # Zero points if it is No Fit
        "Potential Fit": 50, # Fifty points if it is a potential fit
        "Good Fit": 100 # Good Fit gives a Hundred points
    }

    fit_score = 0 # Starting score set to zero

    for label_id, probability in enumerate(probabilities): # Loops through each class probability
        label_name = id2label[label_id] # Converts numeric class ID into text
        weight = score_weights.get(label_name, 0) # Gets the score value from previously defined function
        fit_score += probability * weight # Adds the probability weighted score to the final fit score

    return round(float(fit_score), 2) # Fit score rounded to 2 decimal places

def calculate_confidence(logit_row): # Defines a function to calculate model's confidence level
    probabilities = softmax(logit_row) # Converts raw scores into probabilities
    confidence = np.max(probabilities) # Highest probability is selected as the model's confidence

    if confidence >= 0.75: # Checks if the model is highly confident
        confidence_label = "High" # Confidence is labelled as high
    elif confidence >= 0.50: # Checks if the model is moderately confident
        confidence_label = "Medium" # Confidence is labelled as medium
    else:
        confidence_label = "Low" # Confidence is labelled as low

    return round(float(confidence), 3), confidence_label # Numeric confidence and category is returned

In [22]:
# skill extraction functions

skill_list = [ # A simple list of skills to search among
    "machine learning", "deep learning", "nlp", "data analysis",
    "python", "java", "sql", "excel", "tableau", "power bi",
    "scikit-learn", "tensorflow", "pytorch", "aws", "azure",
    "data visualization", "statistics", "pandas", "numpy",
    "communication", "leadership", "project management",
    "accounting", "etl", "data warehouse", "spark",
    "business intelligence", "powerpoint", "word", "r",
    "customer service", "sales", "marketing", "finance",
    "cloud", "database", "html", "css", "javascript"
]

def extract_skills(text, skills): # Function to find skills inside a text
    text = str(text).lower() # Converts text to lowercase
    found_skills = [] # Empty list to store skills found

    for skill in skills: # Loops through each skill in the predefined list
        pattern = r"\b" + re.escape(skill.lower()) + r"\b" # Creates a regex pattern for skill matching
        if re.search(pattern, text): # Searches for the skill in the text
            found_skills.append(skill) # Adds skills to the found skills list if a match is found 

    return sorted(list(set(found_skills))) # Removes duplicates, sorts skills and returns the final list


def skill_match_explanation(resume_text, job_text, skills): # Function to compare resume skills against job skills
    resume_skills = extract_skills(resume_text, skills) # Extracts skills found in the resume text
    job_skills = extract_skills(job_text, skills) # Extracts skills found in the job description

    matched_skills = sorted(list(set(resume_skills) & set(job_skills))) # Finds the skills that appear in both resume and job description
    missing_skills = sorted(list(set(job_skills) - set(resume_skills))) # Finds the skills that are missing in the resume

    if len(job_skills) == 0: # Checks if no job skills were found
        skill_overlap_score = 0 # Sets the overlap score to 0 if no detected job skills
    else:
        skill_overlap_score = round((len(matched_skills) / len(job_skills)) * 100, 2) # Percentage of job skills matched is calculated

    return { # Returns the results of skill comparision in a dictionary
        "resume_skills": resume_skills,
        "job_skills": job_skills,
        "matched_skills": matched_skills, # Skills that appear in both resume and job description
        "missing_skills": missing_skills, # Skills that are missing in the resume
        "skill_overlap_score": skill_overlap_score # Percentage of job skills matched in the resume
    }

In [23]:
# hybrid score function

def calculate_hybrid_score(transformer_score, skill_overlap_score): # Defines a function to combine model score and skill overlap score
    hybrid_score = (0.70 * transformer_score) + (0.30 * skill_overlap_score) # Combines 70% transformer score with 30% skill overlap score
    return round(hybrid_score, 2) # Final hybrid score is returned, rounded to two decimal places

### Step 21: Creates results table

In [24]:
test_df = pd.DataFrame(test_dataset) # Converts the test dataset into a DataFrame for analysis 

fit_scores = [calculate_fit_score(row) for row in logits] # Calculates the fit score

confidence_scores = [] # Empty list to store confidence scores
confidence_labels = [] # Empty list to store confidence labels

for row in logits: # Loops through each model prediction in the test set
    confidence_score, confidence_label = calculate_confidence(row) # Calculates confidence score and label
    confidence_scores.append(confidence_score) # Adds the confidence score to the list
    confidence_labels.append(confidence_label) # Adds the confidence label to the list

results_df = pd.DataFrame({ # Creates final results table
    "resume_text": test_df["resume_text"], # Original resume text added
    "job_description_text": test_df["job_description_text"], # Original job description text added
    "true_label": true_label_names, # Adds actual label for each example
    "predicted_label": predicted_labels, # Adds predicted label for each example
    "transformer_fit_score": fit_scores, # Adds the 0-100 fit score calculated from transformer probabilities
    "model_confidence": confidence_scores, # Adds the model's numeric confidence score
    "confidence_level": confidence_labels # Adds the text describing model's confidence level
})

results_df.head() # Displays first five rows of the results table

,resume_text,job_description_text,true_label,predicted_label,transformer_fit_score,model_confidence,confidence_level
0,Summary7+ years of experience as a BI develope...,Key Responsibilities:Create intricate wiring n...,No Fit,No Fit,9.59,0.817,High
1,Professional BackgroundAnalyst versed in data ...,Personal development and becoming the best you...,No Fit,No Fit,3.73,0.941,High
2,Executive ProfileDedicated professional with t...,"Location: Tampa, FL\nExp: 7-10 Yrs\nSPOC: Tush...",No Fit,No Fit,2.98,0.952,High
3,"Summarytyee\nHighlightsMicrosoft Excel, Word, ...","Primary Location: Melbourne, Florida\nV-Soft C...",No Fit,No Fit,2.56,0.956,High
4,SummaryEIT certified Engineer and ASTQB Certif...,At Oregon Specialty Group the Accounting & Pay...,No Fit,No Fit,9.10,0.823,High


### Step 22: For each resume - job description pair identifies matched skills, missing skills, and calculates hybrid score

In [25]:
matched_skills_list = [] # Creates an empty list to store skills found in both the resume and job description
missing_skills_list = [] # Creates an empty list to store job skills missing from the resume
resume_skills_list = [] # Creates an empty list for all detected resume skills
job_skills_list = [] # Creates an empty list for all detected job description skills
skill_overlap_scores = [] # Creates an empty list for the skill overlap score
hybrid_scores = [] # Creates an empty list for the final hybrid fit score 

for i in range(len(results_df)): # Loops through every resume-job pair in the results table
    explanation = skill_match_explanation( # Applies the skill comparison function for a resume-job pair
        results_df.loc[i, "resume_text"], # Retrives the resume text from the row being analyzed
        results_df.loc[i, "job_description_text"], # Retrives the job description text from the row being analyzed
        skill_list # Uses the previously defined skill list to search for skills
    )

    resume_skills_list.append(", ".join(explanation["resume_skills"])) # Appends identified resume skills as a comma-separated string
    job_skills_list.append(", ".join(explanation["job_skills"])) # Appends identified job skills as a comma-separated string
    matched_skills_list.append(", ".join(explanation["matched_skills"])) # Appends matched skills as a comma-separated string
    missing_skills_list.append(", ".join(explanation["missing_skills"])) # Appends missing skills as a comma-separated string
    skill_overlap_scores.append(explanation["skill_overlap_score"]) # Appends the skill overlap score for this row

    hybrid_scores.append( # Calculates and stores the final hybrid score
        calculate_hybrid_score( # Combines transformer fit score and skill overlap score
            results_df.loc[i, "transformer_fit_score"], # Gets the transformer's 0-100 fit score for the row being analyzed
            explanation["skill_overlap_score"] # Gets the skill overlap score for the row being analyzed
        )
    )

results_df["resume_skills"] = resume_skills_list # Adds a new column with detected resume skills
results_df["job_skills"] = job_skills_list # Adds a new column with job description skills
results_df["matched_skills"] = matched_skills_list # Adds a new column with matched skills
results_df["missing_skills"] = missing_skills_list # Adds a new column with missing job skills
results_df["skill_overlap_score"] = skill_overlap_scores # Adds a new column with skill overlap scores
results_df["hybrid_fit_score"] = hybrid_scores # Adds a new column with final hybrid fit scores

results_df.head()  # Displays the first five rows of the results table that was built

,resume_text,job_description_text,true_label,predicted_label,transformer_fit_score,model_confidence,confidence_level,resume_skills,job_skills,matched_skills,missing_skills,skill_overlap_score,hybrid_fit_score
0,Summary7+ years of experience as a BI develope...,Key Responsibilities:Create intricate wiring n...,No Fit,No Fit,9.59,0.817,High,"azure, business intelligence, data analysis, d...","communication, excel",excel,communication,50.0,21.71
1,Professional BackgroundAnalyst versed in data ...,Personal development and becoming the best you...,No Fit,No Fit,3.73,0.941,High,"customer service, data analysis, excel, projec...","database, html, javascript",,"database, html, javascript",0.0,2.61
2,Executive ProfileDedicated professional with t...,"Location: Tampa, FL\nExp: 7-10 Yrs\nSPOC: Tush...",No Fit,No Fit,2.98,0.952,High,"accounting, customer service, marketing, r, sales","communication, css, html, javascript",,"communication, css, html, javascript",0.0,2.09
3,"Summarytyee\nHighlightsMicrosoft Excel, Word, ...","Primary Location: Melbourne, Florida\nV-Soft C...",No Fit,No Fit,2.56,0.956,High,"customer service, excel, leadership, marketing...","cloud, communication, java, leadership, sql",leadership,"cloud, communication, java, sql",20.0,7.79
4,SummaryEIT certified Engineer and ASTQB Certif...,At Oregon Specialty Group the Accounting & Pay...,No Fit,No Fit,9.10,0.823,High,"css, database, html, java, javascript, project...","accounting, communication, excel, leadership",,"accounting, communication, excel, leadership",0.0,6.37


### Step 23: Adds a recommendation message and missing skill count

In [26]:
def generate_recommendation(row): # Defines a function that creates the recommendation message
    if row["hybrid_fit_score"] >= 75:
        return "This resume is a strong match for this job description. This candidate's skills are well aligned with the job's required skill set."
    elif row["hybrid_fit_score"] >= 50:
        return "This resume is a possible match for this job description. The candidate has some relevant skills but is also missing some key skills."
    else:
        return "This resume is a weak match. The candidate's resume appears to be missing several important skills."


results_df["recommendation"] = results_df.apply(generate_recommendation, axis=1) #  Applies the recommendation function to every row

results_df["missing_skill_count"] = results_df["missing_skills"].apply( # Creates a new column that counts missing skills
    lambda x: 0 if x == "" else len(x.split(", ")) # Returns 0 for no missing skills, otherwise returns the number of missing skills
)

results_df.head() # Displays the first five rows of the updated results table

,resume_text,job_description_text,true_label,predicted_label,transformer_fit_score,model_confidence,confidence_level,resume_skills,job_skills,matched_skills,missing_skills,skill_overlap_score,hybrid_fit_score,recommendation,missing_skill_count
0,Summary7+ years of experience as a BI develope...,Key Responsibilities:Create intricate wiring n...,No Fit,No Fit,9.59,0.817,High,"azure, business intelligence, data analysis, d...","communication, excel",excel,communication,50.0,21.71,This resume is a weak match. The candidate's r...,1
1,Professional BackgroundAnalyst versed in data ...,Personal development and becoming the best you...,No Fit,No Fit,3.73,0.941,High,"customer service, data analysis, excel, projec...","database, html, javascript",,"database, html, javascript",0.0,2.61,This resume is a weak match. The candidate's r...,3
2,Executive ProfileDedicated professional with t...,"Location: Tampa, FL\nExp: 7-10 Yrs\nSPOC: Tush...",No Fit,No Fit,2.98,0.952,High,"accounting, customer service, marketing, r, sales","communication, css, html, javascript",,"communication, css, html, javascript",0.0,2.09,This resume is a weak match. The candidate's r...,4
3,"Summarytyee\nHighlightsMicrosoft Excel, Word, ...","Primary Location: Melbourne, Florida\nV-Soft C...",No Fit,No Fit,2.56,0.956,High,"customer service, excel, leadership, marketing...","cloud, communication, java, leadership, sql",leadership,"cloud, communication, java, sql",20.0,7.79,This resume is a weak match. The candidate's r...,4
4,SummaryEIT certified Engineer and ASTQB Certif...,At Oregon Specialty Group the Accounting & Pay...,No Fit,No Fit,9.10,0.823,High,"css, database, html, java, javascript, project...","accounting, communication, excel, leadership",,"accounting, communication, excel, leadership",0.0,6.37,This resume is a weak match. The candidate's r...,4


### Step 24: Shows two simple example outputs

In [27]:
sample_index = 0 # Chooses the row from the results table

print("THE TRUE LABEL IS:")
print(results_df.loc[sample_index, "true_label"]) # Prints the correct label from the dataset

print("\nTHE PREDICTED LABEL IS:")
print(results_df.loc[sample_index, "predicted_label"]) # Prints the label predicted by the model

print("\nTHE TRANSFORMER FIT SCORE IS:")
print(results_df.loc[sample_index, "transformer_fit_score"]) # Prints the 0-100 fit score from the transformer model

print("\nTHE SKILL OVERLAP SCORE IS:")
print(results_df.loc[sample_index, "skill_overlap_score"]) # Prints the percentage of job skills found in the resume

print("\nTHE HYBRID FIT SCORE IS:")
print(results_df.loc[sample_index, "hybrid_fit_score"]) # Prints the hybrid score which is the combined transformer and skill-overlap score

print("\nTHE MODEL CONFIDENCE IS:")
print(results_df.loc[sample_index, "model_confidence"]) # Prints the model's confidence score
print(results_df.loc[sample_index, "confidence_level"]) # Prints the confidence category, such as High, Medium, or Low

print("\nTHE MATCHED SKILLS IS:")
print(results_df.loc[sample_index, "matched_skills"]) # Prints the skills that appear in both texts

print("\nTHE MISSING SKILLS IS:")
print(results_df.loc[sample_index, "missing_skills"]) # Prints the job skills not found in the resume

print("\nTHE MISSING SKILL COUNT IS:")
print(results_df.loc[sample_index, "missing_skill_count"]) # Prints the number of job skills missing from the resume

print("\nTHE RECOMMENDATION IS:")
print(results_df.loc[sample_index, "recommendation"]) # Prints a recommendation based on the hybrid score

print("\nTHE RESUME TEXT SAMPLE IS:")
print(results_df.loc[sample_index, "resume_text"][:1200]) # Prints the first 1200 characters of the resume text

print("\nTHE JOB DESCRIPTION SAMPLE IS:")
print(results_df.loc[sample_index, "job_description_text"][:1200]) # Prints the first 1200 characters of the job description text


sample_index = 97 # Chooses the row from the results table

print("THE TRUE LABEL IS:")
print(results_df.loc[sample_index, "true_label"]) # Prints the correct label from the dataset

print("\nTHE PREDICTED LABEL IS:")
print(results_df.loc[sample_index, "predicted_label"]) # Prints the label predicted by the model

print("\nTHE TRANSFORMER FIT SCORE IS:")
print(results_df.loc[sample_index, "transformer_fit_score"]) # Prints the 0-100 fit score from the transformer model

print("\nTHE SKILL OVERLAP SCORE IS:")
print(results_df.loc[sample_index, "skill_overlap_score"]) # Prints the percentage of job skills found in the resume

print("\nTHE HYBRID FIT SCORE IS:")
print(results_df.loc[sample_index, "hybrid_fit_score"]) # Prints the hybrid score which is the combined transformer and skill-overlap score

print("\nTHE MODEL CONFIDENCE IS:")
print(results_df.loc[sample_index, "model_confidence"]) # Prints the model's confidence score
print(results_df.loc[sample_index, "confidence_level"]) # Prints the confidence category, such as High, Medium, or Low

print("\nTHE MATCHED SKILLS IS:")
print(results_df.loc[sample_index, "matched_skills"]) # Prints the skills that appear in both texts

print("\nTHE MISSING SKILLS IS:")
print(results_df.loc[sample_index, "missing_skills"]) # Prints the job skills not found in the resume

print("\nTHE MISSING SKILL COUNT IS:")
print(results_df.loc[sample_index, "missing_skill_count"]) # Prints the number of job skills missing from the resume

print("\nTHE RECOMMENDATION IS:")
print(results_df.loc[sample_index, "recommendation"]) # Prints a recommendation based on the hybrid score

print("\nTHE RESUME TEXT SAMPLE IS:")
print(results_df.loc[sample_index, "resume_text"][:1200]) # Prints the first 1200 characters of the resume text

print("\nTHE JOB DESCRIPTION SAMPLE IS:")
print(results_df.loc[sample_index, "job_description_text"][:1200]) # Prints the first 1200 characters of the job description text

THE TRUE LABEL IS:
No Fit

THE PREDICTED LABEL IS:
No Fit

THE TRANSFORMER FIT SCORE IS:
9.59

THE SKILL OVERLAP SCORE IS:
50.0

THE HYBRID FIT SCORE IS:
21.71

THE MODEL CONFIDENCE IS:
0.817
High

THE MATCHED SKILLS IS:
excel

THE MISSING SKILLS IS:
communication

THE MISSING SKILL COUNT IS:
1

THE RECOMMENDATION IS:
This resume is a weak match. The candidate's resume appears to be missing several important skills.

THE RESUME TEXT SAMPLE IS:
Summary7+ years of experience as a BI developer with a proven track record in Business Intelligence (BI), Data Warehouse (DWH) and Data Analytics related consulting projects.Proven ability to identify business needs and develop valuable solutions to drive accuracy and process efficiency.Experienced in developing, implementing, documenting, monitoring and maintaining the data warehouse extracts, transformations and ETL process in various industries like Financial, Health Care and Retail Industry withSales, Marketing, Inventory Management, Supply C

### Step 25: Displays top 10 best resume-job description matches based on hybrid score

In [28]:
top_10_matches = results_df.sort_values( # Sorts the results table by the chosen column
    by="hybrid_fit_score", # Applies hybrid fit score as the sorting column
    ascending=False # Sorts from highest score to lowest score
).head(10) # Chooses the top 10 highest-scoring resume-job matches

top_10_matches[
    [
        "true_label", # The actual label
        "predicted_label", # The model's predicted label
        "transformer_fit_score", # The fit score
        "skill_overlap_score", # The percentage of job skills found in the resume
        "hybrid_fit_score", # The final combined score
        "model_confidence", # The model confidence
        "confidence_level", # The confidence category
        "matched_skills",  # The skills found in both the resume and job description
        "missing_skills", # The job skills that were not found in the resume
        "missing_skill_count", # The number of missing job skills
        "recommendation" # The recommendation message
    ]
]

,true_label,predicted_label,transformer_fit_score,skill_overlap_score,hybrid_fit_score,model_confidence,confidence_level,matched_skills,missing_skills,missing_skill_count,recommendation
1227,Potential Fit,Good Fit,93.16,100.0,95.21,0.914,High,"azure, cloud, communication",,0,This resume is a strong match for this job des...
722,No Fit,Good Fit,92.78,100.0,94.95,0.903,High,"css, html, java, javascript",,0,This resume is a strong match for this job des...
581,No Fit,Good Fit,88.61,100.0,92.03,0.859,High,"css, html, java, javascript",,0,This resume is a strong match for this job des...
1454,Good Fit,Good Fit,85.35,100.0,89.74,0.793,High,accounting,,0,This resume is a strong match for this job des...
1280,Potential Fit,Good Fit,85.25,100.0,89.68,0.824,High,"css, html, java, javascript",,0,This resume is a strong match for this job des...
430,No Fit,Good Fit,83.50,100.0,88.45,0.779,High,"aws, azure, cloud, communication",,0,This resume is a strong match for this job des...
935,Potential Fit,Good Fit,91.31,80.0,87.92,0.882,High,"azure, cloud, java, sql",aws,1,This resume is a strong match for this job des...
1446,Good Fit,Good Fit,91.12,80.0,87.78,0.889,High,"aws, cloud, java, sql",azure,1,This resume is a strong match for this job des...
1624,Good Fit,Good Fit,82.27,100.0,87.59,0.751,High,"communication, css, html, javascript",,0,This resume is a strong match for this job des...
1310,Good Fit,Good Fit,89.12,80.0,86.38,0.868,High,"azure, cloud, java, sql",aws,1,This resume is a strong match for this job des...


### Step 26: Saves results

In [29]:
os.makedirs("outputs/transformer_results", exist_ok=True) # Creates an output folder to save the results

results_df.to_csv( # Saves the final results table in a CSV file
    "outputs/transformer_results/DistilBERT_transformer_predictions.csv", # Sets the filename and path for the saved CSV
    index=False # Does not include the DataFrame row index in the file
)

print("Saved predictions to outputs/transformer_results/DistilBERT_transformer_predictions.csv") # Prints a confirmation message

Saved predictions to outputs/transformer_results/DistilBERT_transformer_predictions.csv


### Step 27: Saves model

In [30]:
os.makedirs("models/trained_transformer_model", exist_ok=True) # Creates a folder for the model

trainer.save_model("models/trained_transformer_model") # Saves the trained transformer model files
tokenizer.save_pretrained("models/trained_transformer_model") # Also saves the tokenizer files needed to process text the same way

print("Saved model to models/trained_transformer_model")  # Prints a confirmation message

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model to models/trained_transformer_model


### Step 28: A short summary for the README

In [31]:
summary = f"""
Transformer Model Summary

Model used: {model_checkpoint}

Highlights:
Trained on the full dataset with class weighting.
Added weighted and macro metrics.
Added confidence, matched skills, missing skills, hybrid scores, and recommendations.

Evaluation Results:
Accuracy: {test_results.get('eval_accuracy')}
Weighted F1-score: {test_results.get('eval_f1_weighted')}
Macro F1-score: {test_results.get('eval_f1_macro')}
"""

print(summary)


Transformer Model Summary

Model used: distilbert-base-uncased

Highlights:
Trained on the full dataset with class weighting.
Added weighted and macro metrics.
Added confidence, matched skills, missing skills, hybrid scores, and recommendations.

Evaluation Results:
Accuracy: 0.47868106878908473
Weighted F1-score: 0.44405648053300223
Macro F1-score: 0.3877684525157646



### References
Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, L., & Polosukhin, I. (2017, June 12). Attention Is All You Need. Cornell University. https://arxiv.org/abs/1706.03762

‌Devlin, J., Chang, M.-W., Lee, K., & Toutanova, K. (2018, October 11). BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding. ArXiv. https://arxiv.org/abs/1810.04805

‌Sanh, V., Debut, L., Chaumond, J., & Wolf, T. (2019). DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter. ArXiv.org. https://arxiv.org/abs/1910.01108

‌Wolf, T., Debut, L., Sanh, V., Chaumond, J., Delangue, C., Moi, A., Cistac, P., Rault, T., Louf, R., Funtowicz, M., Davison, J., Shleifer, S., von Platen, P., Ma, C., Jernite, Y., Plu, J., Xu, C., Le Scao, T., Gugger, S., & Drame, M. (2020). Transformers: State-of-the-Art Natural Language Processing. Proceedings of the 2020 Conference on Empirical Methods in Natural Language Processing: System Demonstrations, 38–45. https://doi.org/10.18653/v1/2020.emnlp-demos.6

‌Datasets. (n.d.). Huggingface.co. https://huggingface.co/docs/datasets/en/index

Namuangtoun, C. (n.d.). Resume-job-description-fit [Data set]. Hugging Face. https://huggingface.co/datasets/cnamuangtoun/resume-job-description-fit

‌Paszke, A., Gross, S., Massa, F., Lerer, A., Bradbury, J., Chanan, G., Killeen, T., Lin, Z., Gimelshein, N., Antiga, L., Desmaison, A., Köpf, A., Yang, E., DeVito, Z., Raison, M., Tejani, A., Chilamkurthy, S., Steiner, B., Fang, L., & Bai, J. (2019). PyTorch: An Imperative Style, High-Performance Deep Learning Library. ArXiv.org. https://arxiv.org/abs/1912.01703

‌Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., & Duchesnay, É. (2011). Scikit-learn: Machine Learning in Python. Journal of Machine Learning Research, 12(85), 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html

‌McKinney, W. (2010). Data Structures for Statistical Computing in Python. Proceedings of the 9th Python in Science Conference, 445. https://doi.org/10.25080/majora-92bf1922-00a

‌Harris, C. R., Millman, K. J., van der Walt, S. J., Gommers, R., Virtanen, P., Cournapeau, D., Wieser, E., Taylor, J., Berg, S., Smith, N. J., Kern, R., Picus, M., Hoyer, S., van Kerkwijk, M. H., Brett, M., Haldane, A., del Río, J. F., Wiebe, M., Peterson, P., & Gérard-Marchant, P. (2020). Array Programming with NumPy. Nature, 585(7825), 357–362. https://doi.org/10.1038/s41586-020-2649-2

‌Cui, Y., Jia, M., Lin, T.-Y., Song, Y., & Belongie, S. (2019). Class-Balanced Loss Based on Effective Number of Samples. ArXiv:1901.05555 [Cs]. https://arxiv.org/abs/1901.05555

‌Loshchilov, I., & Hutter, F. (2017). Decoupled Weight Decay Regularization. Arxiv.org. https://arxiv.org/abs/1711.05101

‌Guo, C., Pleiss, G., Sun, Y., & Weinberger, K. Q. (2017, July 17). On Calibration of Modern Neural Networks. Proceedings.mlr.press; PMLR. https://proceedings.mlr.press/v70/guo17a.html

‌Yu, X., Zhang, J., & Yu, Z. (2024). ConFit: Improving Resume-Job Matching using Data Augmentation and Contrastive Learning. ArXiv.org. https://arxiv.org/abs/2401.16349